In [ ]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lightglue import viz2d

from uavcalibration.calibration import Calibration
from uavcalibration.datasets import VisLocDataset
from uavcalibration.transform import *

dataset = VisLocDataset(r"E:\Archive\DataSet\Image\UAV\UAV_VisLoc_example")

In [ ]:
uav_data = dataset[50]
calibration = Calibration(uav_data.uav_image)
calibration.coarse_calibrate(**asdict(uav_data))
src_shape = uav_data.uav_image.shape
calibration.transform.adjust_shape(src_shape=(src_shape[1],src_shape[0]))
uav_image = calibration.calibrated_image

In [ ]:
h, w = calibration.uav_image.shape[:2]
corner = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float64)
corner_utm = cv2.perspectiveTransform(
    corner.reshape(1, -1, 2), calibration.transform.combined.mat
).reshape(-1, 2)
transformer = Transformer.from_crs(calibration.transform.crs.crs, "epsg:4326",always_xy=True)
corner_lonlat = np.stack(transformer.transform(corner_utm[:, 0], corner_utm[:, 1]), 1)
corner_lonlat = np.stack([corner_lonlat.min(0), corner_lonlat.max(0)])
print(corner_lonlat.ravel())
# fetch satellite image here

In [ ]:
calibration.fine_calibrate(satellite_image=uav_data.satellite_image, satellite_crs=CRSTransform(uav_data.satellite_transform))
# viz2d.plot_images([calibration.calibrated_image, uav_data.satellite_image])

In [ ]:
# 全局变量存储鼠标位置
uav_pos = (0, 0)
satellite_pos = (0, 0)
instance = 0

# 鼠标回调函数
def mouse_callback(event, x, y, flags, param):
    global uav_pos, satellite_pos, instance
    if event == cv2.EVENT_MOUSEMOVE:  # 鼠标移动事件
        uav_pos = (x, y)
        satellite_pos = tuple(
            calibration.transform.apply(np.array(uav_pos, np.float64))
            .ravel()
            .astype(int)
        )
    elif 81 <= event <= 84:
        if event in [81, 82]:  # left,up
            instance -= 1
        elif event in [83, 84]:  # right,down
            instance += 1
        # TODO:update

# 创建窗口并绑定回调函数
cv2.namedWindow("UAV Image", cv2.WINDOW_KEEPRATIO)
cv2.setMouseCallback("UAV Image", mouse_callback)
cv2.namedWindow("Satellite Image", cv2.WINDOW_KEEPRATIO)

size = 10
while True:
    # 在图像上绘制坐标（可选）
    uav_image = uav_data.uav_image[..., ::-1].copy()
    satellite_image = uav_data.satellite_image[..., ::-1].copy()
    cv2.putText(
        uav_image,
        f"Position: {uav_pos}",
        (size * 5, size * 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        size * 0.3,
        (0, 255, 0),
        size,
    )
    cv2.circle(satellite_image, satellite_pos, size, (0, 255, 0), size)
    # 显示图像
    cv2.imshow("UAV Image", uav_image)
    cv2.imshow("Satellite Image", satellite_image)
    # 按ESC键退出
    if cv2.waitKey(1) == 27:
        break
cv2.destroyAllWindows()